In [1]:
# =====================================================================
# Cell 1 — Install packages
# =====================================================================
!pip install -q torch torchvision timm wandb huggingface_hub tqdm numpy pyyaml pandas matplotlib


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# =====================================================================
# Cell 2 — Imports and global config
# =====================================================================
import os, json, time, zipfile, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision import datasets

from timm.data import Mixup, create_transform
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import CosineLRScheduler
from timm.models.layers import DropPath, trunc_normal_

from torch.profiler import profile, ProfilerActivity
from huggingface_hub import hf_hub_download, login, whoami
from tqdm.auto import tqdm

# ------------------- CONFIG -------------------
DATA_ROOT       = "./data"
RAW_ROOT        = "./data/imagenet100_raw"
HF_REPO         = "asafaa/imagent100"
TRAIN_PARTS     = 5
VAL_PARTS       = 5
OUTPUT_DIR      = "./outputs/phase0"

EPOCHS          = 100
BATCH_SIZE      = 128
LR              = 2e-3         # ViG-Ti recipe scaled for batch=128
WEIGHT_DECAY    = 0.05
WARMUP_EPOCHS   = 5            # shorter warmup; model learns faster
GRAD_CLIP       = 1.0
NUM_WORKERS     = 8
NUM_CLASSES     = 100
SEED            = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

os.makedirs(DATA_ROOT,  exist_ok=True)
os.makedirs(RAW_ROOT,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device:     {DEVICE}")
print(f"Output dir: {OUTPUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")

Device:     cuda
Output dir: ./outputs/phase0
GPU:        NVIDIA H200


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# =====================================================================
# Cell 3 — HuggingFace login
# =====================================================================
HF_TOKEN = "hf_key"
login(token=HF_TOKEN)
try:
    print(f"✓ Logged in as: {whoami()['name']}")
except Exception as e:
    print(f"⚠ whoami failed: {e}")

✓ Logged in as: Ranjan637


In [ ]:
# =====================================================================
# Cell 4 — Download ImageNet-100 zip chunks + Labels.json
# =====================================================================
def download_all_chunks(repo_id, raw_root, train_parts=5, val_parts=5):
    os.makedirs(raw_root, exist_ok=True)
    downloaded = []
    for split, n_parts in [("train", train_parts), ("val", val_parts)]:
        for part in range(1, n_parts + 1):
            fn = f"imagenet100_{split}_part{part}.zip"
            print(f"Downloading {fn} ...")
            try:
                p = hf_hub_download(repo_id=repo_id, filename=fn,
                                    repo_type="dataset", local_dir=raw_root)
                downloaded.append(p)
                print(f"  ✓ {p}")
            except Exception as e:
                print(f"  ✗ {e}")
    try:
        lp = hf_hub_download(repo_id=repo_id, filename="Labels.json",
                             repo_type="dataset", local_dir=raw_root)
        downloaded.append(lp)
        print(f"  ✓ Labels.json -> {lp}")
    except Exception as e:
        print(f"  ⚠ Labels.json: {e}")
    return downloaded

downloaded_files = download_all_chunks(HF_REPO, RAW_ROOT, TRAIN_PARTS, VAL_PARTS)
print(f"\nTotal files downloaded: {len(downloaded_files)}")

In [ ]:
# =====================================================================
# Cell 5 — Extract and merge into ImageFolder tree
# =====================================================================
def extract_and_merge(raw_root, data_root, train_parts=5, val_parts=5):
    tmp_dir = os.path.join(raw_root, "_extract_tmp")
    os.makedirs(tmp_dir, exist_ok=True)
    for split, n_parts in [("train", train_parts), ("val", val_parts)]:
        split_dir = os.path.join(data_root, split)
        os.makedirs(split_dir, exist_ok=True)
        for part in range(1, n_parts + 1):
            zn = f"imagenet100_{split}_part{part}.zip"
            zp = os.path.join(raw_root, zn)
            if not os.path.exists(zp):
                print(f"  ⚠ Missing {zn}"); continue
            et = os.path.join(tmp_dir, f"{split}_part{part}")
            os.makedirs(et, exist_ok=True)
            print(f"Extracting {zn}")
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(et)
            for root, dirs, files in os.walk(et):
                if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
                    cls_name = os.path.basename(root)
                    dest = os.path.join(split_dir, cls_name)
                    os.makedirs(dest, exist_ok=True)
                    for f in files:
                        if f.lower().endswith((".jpg", ".jpeg", ".png")):
                            s = os.path.join(root, f); d = os.path.join(dest, f)
                            if not os.path.exists(d): shutil.move(s, d)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print("Extraction + merge complete.")

extract_and_merge(RAW_ROOT, DATA_ROOT, TRAIN_PARTS, VAL_PARTS)

In [4]:
# =====================================================================
# Cell 6 — Verify dataset
# =====================================================================
def verify_dataset(data_root):
    for split in ["train", "val"]:
        d = os.path.join(data_root, split)
        if not os.path.isdir(d):
            print(f"✗ {d} missing"); continue
        cls = sorted([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
        n = sum(len(fs) for _, _, fs in os.walk(d))
        print(f"{split}: {len(cls)} classes, {n} images")
    tr = set(os.listdir(os.path.join(data_root, "train")))
    vl = set(os.listdir(os.path.join(data_root, "val")))
    print("✓ Train/val class folders match." if tr == vl else "⚠ Class folders differ!")

verify_dataset(DATA_ROOT)

train: 100 classes, 128982 images
val: 100 classes, 5000 images
✓ Train/val class folders match.


In [5]:
# =====================================================================
# Cell 7 — CORRECTED MODEL: LayerNorm2d instead of BatchNorm2d
# =====================================================================
# Critical fix: LayerNorm has NO running statistics, so eval == train
# numerically. This eliminates the val_loss explosion.
# ---------------------------------------------------------------------

class LayerNorm2d(nn.Module):
    """LayerNorm over the channel dim for (B, C, H, W) tensors."""
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias   = nn.Parameter(torch.zeros(channels))
        self.eps    = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


class MRConv2d(nn.Module):
    """Max-Relative Graph Convolution (stable, project-then-aggregate)."""
    def __init__(self, in_channels, out_channels, act='relu', bias=True):
        super().__init__()
        self.act = nn.ReLU() if act == 'relu' else nn.GELU()
        self.fc  = nn.Linear(in_channels, out_channels, bias=bias)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x, edge_index):
        N = x.size(0)
        row, col = edge_index
        x_j = x[row] - x[col]
        x_j = self.act(x_j)
        x_j = self.fc(x_j)
        out = torch.zeros(N, x_j.size(-1), device=x_j.device, dtype=x_j.dtype)
        out.index_add_(0, col, x_j)
        return self.norm(out)


class Grapher(nn.Module):
    """Grapher: kNN + graph conv + FFN with LayerNorm everywhere."""

    def __init__(self, in_channels, kernel_size=9, dilation=1, drop_path=0.0):
        super().__init__()
        self.k = kernel_size
        self.dilation = dilation

        # Pre-processing 1x1 conv + LayerNorm
        self.fc1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1, bias=False),
            LayerNorm2d(in_channels),
        )
        # Graph convolution
        self.graph_conv = MRConv2d(in_channels, in_channels * 2, act='relu')
        # FFN (post-processing MLP) + LayerNorm
        self.ffn = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels * 4, 1, bias=False),
            LayerNorm2d(in_channels * 4),
            nn.GELU(),
            nn.Conv2d(in_channels * 4, in_channels, 1, bias=False),
            LayerNorm2d(in_channels),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def _build_knn_graph(self, x):
        B, C, H, W = x.shape
        N = H * W
        k_eff = min(self.k, N - 1)
        if k_eff < 1:
            return torch.empty((2, 0), dtype=torch.long, device=x.device), B, N

        with torch.amp.autocast('cuda', enabled=False):
            x_flat = x.permute(0, 2, 3, 1).reshape(B, N, C).float()
            x_norm = F.normalize(x_flat, dim=-1, eps=1e-6)
            sim    = torch.bmm(x_norm, x_norm.transpose(1, 2))
            sim.diagonal(dim1=1, dim2=2).fill_(-float('inf'))
            _, knn_idx = sim.topk(k_eff, dim=-1)

        base = torch.arange(B, device=x.device).view(B, 1, 1) * N
        row  = (torch.arange(N, device=x.device).view(1, N, 1) + base).expand(B, N, k_eff)
        col  = knn_idx + base.expand(B, N, k_eff)
        edge_index = torch.stack([row.reshape(-1), col.reshape(-1)], dim=0)
        return edge_index, B, N

    def forward(self, x):
        B, C, H, W = x.shape
        # Pre-processing
        x_pre = self.fc1(x)
        # Graph construction
        edge_index, B, N = self._build_knn_graph(x_pre)
        # Graph convolution
        x_flat = x_pre.permute(0, 2, 3, 1).reshape(B * N, C)
        x_conv = self.graph_conv(x_flat, edge_index)
        x_conv = x_conv.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        # FFN
        x_ffn = self.ffn(x_conv)
        # Residual
        return self.drop_path(x_ffn) + x


class PyramidViG(nn.Module):
    """Pyramid ViG-Ti: 4 stages, depths (2,2,6,2), embed_dim=48."""

    def __init__(self, num_classes=1000, in_chans=3, embed_dim=48,
                 depths=(2, 2, 6, 2), drop_path_rate=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths

        # Stem: patchify with LayerNorm
        self.stem = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, kernel_size=16, stride=16, bias=False),
            LayerNorm2d(embed_dim),
            nn.GELU(),
        )

        self.stages = nn.ModuleList()
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i, depth in enumerate(depths):
            dim = embed_dim * (2 ** i)
            if i > 0:
                downsample = nn.Sequential(
                    nn.Conv2d(embed_dim * (2 ** (i - 1)), dim,
                              kernel_size=2, stride=2, bias=False),
                    LayerNorm2d(dim),
                )
            else:
                downsample = nn.Identity()
            stage = nn.Sequential(
                downsample,
                *[Grapher(dim, kernel_size=9, drop_path=dpr[cur + j])
                  for j in range(depth)]
            )
            self.stages.append(stage)
            cur += depth

        # Final norm + head
        self.norm = nn.LayerNorm(embed_dim * 8)
        self.head = nn.Linear(embed_dim * 8, num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, LayerNorm2d):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        x = self.stem(x)
        for stage in self.stages:
            x = stage(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = self.norm(x)
        x = self.head(x)
        return x


def pyramid_vig_ti(num_classes=100):
    return PyramidViG(num_classes=num_classes, embed_dim=48,
                      depths=(2, 2, 6, 2), drop_path_rate=0.1)


_m = pyramid_vig_ti(NUM_CLASSES)
print(f"Pyramid ViG-Ti params: {sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M")
del _m; torch.cuda.empty_cache()

Pyramid ViG-Ti params: 8.59M


In [6]:
# =====================================================================
# Cell 8 — Data loaders
# =====================================================================
def build_loaders(data_root, batch_size=128, num_workers=8):
    train_transform = create_transform(
        input_size=224, is_training=True,
        color_jitter=0.4, auto_augment='rand-m9-mstd0.5-inc1',
        interpolation='bicubic', re_prob=0.25, re_mode='pixel', re_count=1,
    )
    val_transform = create_transform(
        input_size=224, is_training=False,
        interpolation='bicubic', crop_pct=0.95,
    )
    tr_ds = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_transform)
    vl_ds = datasets.ImageFolder(os.path.join(data_root, "val"),   transform=val_transform)

    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=num_workers, pin_memory=True,
                       drop_last=True, persistent_workers=(num_workers > 0))
    vl_ld = DataLoader(vl_ds, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True,
                       persistent_workers=(num_workers > 0))
    return tr_ld, vl_ld, len(tr_ds.classes)

train_loader, val_loader, n_classes = build_loaders(DATA_ROOT, BATCH_SIZE, NUM_WORKERS)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Classes: {n_classes}")

Train batches: 1007 | Val batches: 40 | Classes: 100


In [7]:
# =====================================================================
# Cell 9 — Sanity check: no NaN, correct shape, eval matches train
# =====================================================================
def sanity_check(model, loader, device):
    model.train()
    xb, _ = next(iter(loader))
    xb = xb.to(device)
    with torch.no_grad(), autocast('cuda'):
        out_train = model(xb)
    model.eval()
    with torch.no_grad(), autocast('cuda'):
        out_eval = model(xb)
    ot, oe = out_train.float(), out_eval.float()
    print(f"Output shape : {tuple(ot.shape)}")
    print(f"Train mode   : mean={ot.mean():.4f}  std={ot.std():.4f}  nan={torch.isnan(ot).any().item()}")
    print(f"Eval mode    : mean={oe.mean():.4f}  std={oe.std():.4f}  nan={torch.isnan(oe).any().item()}")
    diff = (ot - oe).abs().mean().item()
    print(f"Train-eval diff: {diff:.6f}  (should be ~0 with LayerNorm)")

_m = pyramid_vig_ti(num_classes=NUM_CLASSES).to(DEVICE)
sanity_check(_m, train_loader, DEVICE)
del _m; torch.cuda.empty_cache()

Output shape : (128, 100)
Train mode   : mean=-0.0154  std=0.3826  nan=False
Eval mode    : mean=-0.0158  std=0.3840  nan=False
Train-eval diff: 0.098085  (should be ~0 with LayerNorm)


In [8]:
# =====================================================================
# Cell 10 — Experiment 0.1: Training + validation
# =====================================================================
def train_one_epoch(model, loader, optimizer, scaler, criterion, mixup_fn, device):
    model.train()
    total_loss, total = 0.0, 0
    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if mixup_fn is not None:
            images, targets = mixup_fn(images, targets)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    return total_loss / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="Val", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        correct += pred.eq(targets).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def run_experiment_0_1():
    print("=" * 80)
    print("EXPERIMENT 0.1: Reproduce Pyramid ViG-Ti on ImageNet-100")
    print("=" * 80)

    model = pyramid_vig_ti(num_classes=NUM_CLASSES).to(DEVICE)
    print(f"Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

    mixup_fn = Mixup(mixup_alpha=0.8, cutmix_alpha=1.0, cutmix_minmax=None,
                     prob=1.0, switch_prob=0.5, mode='batch',
                     label_smoothing=0.1, num_classes=NUM_CLASSES)
    train_criterion = SoftTargetCrossEntropy()
    val_criterion   = nn.CrossEntropyLoss()

    # No weight decay on norm params (standard practice)
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim <= 1 or 'norm' in n.lower() or 'bias' in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    optimizer = optim.AdamW([
        {'params': decay,    'weight_decay': WEIGHT_DECAY},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=LR)

    scaler = GradScaler('cuda')
    scheduler = CosineLRScheduler(optimizer, t_initial=EPOCHS, lr_min=1e-6,
                                  warmup_t=WARMUP_EPOCHS, warmup_lr_init=1e-6)

    best_acc, history = 0.0, []
    for epoch in range(EPOCHS):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, scaler,
                                  train_criterion, mixup_fn, DEVICE)
        vl_loss, vl_acc = validate(model, val_loader, val_criterion, DEVICE)
        scheduler.step(epoch)
        elapsed = time.time() - t0
        history.append(dict(epoch=epoch+1, train_loss=tr_loss, val_loss=vl_loss,
                            val_acc=vl_acc, lr=optimizer.param_groups[0]['lr'],
                            time=elapsed))
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss {tr_loss:.4f} | "
              f"Val Loss {vl_loss:.4f} Acc {vl_acc:.4f} | {elapsed:.1f}s")
        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_vig_ti.pth"))
            print(f"  -> New best: {best_acc:.4f}")

    print(f"\nBest validation accuracy: {best_acc:.4f}")
    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
    return best_acc, history

In [9]:
# =====================================================================
# Cell 11 — Experiment 0.2: Profiling (no training)
# =====================================================================
def profile_model(model, input_size=(1, 3, 224, 224), device='cuda', warmup=10, runs=50):
    model.eval().to(device)
    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        for _ in range(warmup): _ = model(dummy)
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 record_shapes=True, profile_memory=True) as prof:
        with torch.no_grad():
            for _ in range(runs): _ = model(dummy)

    print("\n" + "=" * 80)
    print(f"PROFILING: ViG-Ti | input={input_size}")
    print("=" * 80)
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))

    # Component-level timing
    timings = {'knn': [], 'graph_conv': [], 'ffn': [], 'total': []}
    orig = Grapher.forward
    def instr(self, x):
        torch.cuda.synchronize()
        e = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        e[0].record()
        B, C, H, W = x.shape
        x_pre = self.fc1(x)
        edge_index, B, N = self._build_knn_graph(x_pre)
        e[1].record()
        x_flat = x_pre.permute(0, 2, 3, 1).reshape(B * N, C)
        x_conv = self.graph_conv(x_flat, edge_index)
        x_conv = x_conv.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        e[2].record()
        x_ffn = self.ffn(x_conv)
        e[3].record()
        x_out = self.drop_path(x_ffn) + x
        e[4].record()
        torch.cuda.synchronize()
        timings['knn'].append(e[0].elapsed_time(e[1]))
        timings['graph_conv'].append(e[1].elapsed_time(e[2]))
        timings['ffn'].append(e[2].elapsed_time(e[3]))
        timings['total'].append(e[0].elapsed_time(e[4]))
        return x_out
    Grapher.forward = instr
    with torch.no_grad():
        for _ in range(runs): _ = model(dummy)
    torch.cuda.synchronize()
    Grapher.forward = orig

    print("\n--- Average per-Grapher-layer timings (ms) ---")
    for k in ['knn', 'graph_conv', 'ffn', 'total']:
        arr = np.array(timings[k])
        print(f"  {k:12s}: {arr.mean():.3f} ms (std {arr.std():.3f})")

    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(): _ = model(dummy)
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")
    return prof, timings


def run_experiment_0_2():
    print("=" * 80)
    print("EXPERIMENT 0.2: Profile ViG-Ti")
    print("=" * 80)
    model = pyramid_vig_ti(num_classes=NUM_CLASSES)
    print(f"Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")
    print("\n>>> Batch size 1")
    profile_model(model, (1, 3, 224, 224), DEVICE)
    print("\n>>> Batch size 32")
    profile_model(model, (32, 3, 224, 224), DEVICE)

In [10]:
# =====================================================================
# Cell 12 — RUN Phase 0
# =====================================================================
RUN_TRAINING  = True
RUN_PROFILING = False

if RUN_TRAINING:
    best_acc, history = run_experiment_0_1()

if RUN_PROFILING:
    prof, timings = run_experiment_0_2()

EXPERIMENT 0.1: Reproduce Pyramid ViG-Ti on ImageNet-100
Model params: 8.59M


Epoch 1/100 | Train Loss 4.6210 | Val Loss 4.6127 Acc 0.0100 | 78.2s
  -> New best: 0.0100


Epoch 2/100 | Train Loss 4.6122 | Val Loss 4.6084 Acc 0.0106 | 76.0s
  -> New best: 0.0106


Epoch 3/100 | Train Loss 4.6259 | Val Loss 4.6115 Acc 0.0100 | 70.1s


Epoch 4/100 | Train Loss 4.6140 | Val Loss 4.6087 Acc 0.0100 | 70.1s


Epoch 5/100 | Train Loss 4.6086 | Val Loss 4.6063 Acc 0.0100 | 74.4s


... (91 outputs left out)

Epoch 97/100 | Train Loss nan | Val Loss nan Acc 0.0100 | 75.7s


Epoch 98/100 | Train Loss nan | Val Loss nan Acc 0.0100 | 73.9s


Epoch 99/100 | Train Loss nan | Val Loss nan Acc 0.0100 | 72.5s


Epoch 100/100 | Train Loss nan | Val Loss nan Acc 0.0100 | 76.0s

Best validation accuracy: 0.0106


In [ ]:
# =====================================================================
# Cell 13 — Summary plots
# =====================================================================
print("=" * 80); print("PHASE 0 — SUMMARY"); print("=" * 80)

if RUN_TRAINING and "best_acc" in globals():
    print(f"\n[0.1] Best val acc: {best_acc:.4f}")
    hp = os.path.join(OUTPUT_DIR, "training_history.csv")
    if os.path.exists(hp):
        df = pd.read_csv(hp)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(df["epoch"], df["val_acc"], marker="s", ms=3, color="tab:blue")
        axes[0].set_title("Val Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)
        axes[1].plot(df["epoch"], df["train_loss"], label="Train", marker="o", ms=3)
        axes[1].plot(df["epoch"], df["val_loss"],   label="Val",   marker="s", ms=3)
        axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
        axes[2].plot(df["epoch"], df["lr"], color="tab:green")
        axes[2].set_title("LR"); axes[2].set_xlabel("Epoch"); axes[2].grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "phase0_curves.png"), dpi=150)
        plt.show()